# Representation Demystifies Flat Minima
## Notebook 43: SAM vs Representation

Earlier notebooks showed:

> Same function. Same predictions. Different Hessian sharpness.

This notebook asks what that means for SAM-style methods. SAM is often interpreted as pushing models toward flatter minima. If parameter-space flatness depends on representation, then SAM may be better understood as a robustness method whose useful signal should be measured closer to function space.

## Output files

Running this notebook creates:

- `figures/43_sam_penalty_vs_scale.png`
- `figures/43_same_radius_different_function_change.png`
- `figures/43_parameter_vs_function_space_sam.png`
- `figures/43_sam_interpretation_shift.png`
- `results/43_sam_vs_representation_summary.csv`
- `results/43_sam_vs_representation_summary.json`
- `downloads/43_sam_vs_representation_outputs.zip`

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
DOWNLOADS = ROOT / "downloads"

for path in [FIGURES, RESULTS, DOWNLOADS]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

## 1. SAM intuition

A simplified SAM-style objective is:

\[
\min_\theta \max_{\|\epsilon\| \leq \rho} L(\theta + \epsilon)
\]

Informally:

> Do not only minimize loss at the point. Minimize loss around the point.

Classical interpretation:

\[
\text{SAM} \rightarrow \text{flatter minima} \rightarrow \text{better generalization}
\]

Reframed interpretation:

\[
\text{SAM} \rightarrow \text{robustness proxy} \rightarrow \text{possibly better generalization}
\]

## 2. Same function under reparameterization

Use the toy model:

\[
f(x;w)=wx
\]

with target \(y=x\), and reparameterization:

\[
w=su
\]

At \(u^*=1/s\), every scale computes the same function:

\[
f(x;su^*)=x
\]

In [ ]:
x = np.linspace(-1, 1, 800)
y = x
ex2 = np.mean(x**2)

scales = np.array([0.25, 0.5, 1, 2, 4, 8])
rho_u = 0.05
delta_w = 0.05

rows = []

for s in scales:
    u_star = 1 / s
    w_effective = s * u_star
    base_loss = np.mean((w_effective * x - y) ** 2)

    eps_grid = np.linspace(-rho_u, rho_u, 401)
    losses_u = []
    function_changes = []

    for eps in eps_grid:
        u_perturbed = u_star + eps
        w_perturbed = s * u_perturbed
        y_hat = w_perturbed * x
        losses_u.append(np.mean((y_hat - y) ** 2))
        function_changes.append(abs(w_perturbed - w_effective))

    sam_penalty_u = max(losses_u)
    max_function_change = max(function_changes)

    w_grid = w_effective + np.linspace(-delta_w, delta_w, 401)
    losses_w = [np.mean((w_perturbed * x - y) ** 2) for w_perturbed in w_grid]
    sam_penalty_function = max(losses_w)

    rows.append({
        "scale_s": float(s),
        "u_star": float(u_star),
        "w_effective": float(w_effective),
        "base_loss": float(base_loss),
        "rho_u": float(rho_u),
        "parameter_space_sam_penalty": float(sam_penalty_u),
        "max_function_change_from_u_radius": float(max_function_change),
        "delta_w": float(delta_w),
        "function_space_penalty": float(sam_penalty_function),
        "penalty_ratio_parameter_to_function": float(sam_penalty_u / sam_penalty_function)
    })

results = pd.DataFrame(rows)
results

## 3. Parameter-space SAM penalty changes with representation

A fixed perturbation in \(u\)-space maps to:

\[
\Delta w = s\epsilon_u
\]

so the same coordinate radius can produce different functional changes.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results["scale_s"], results["parameter_space_sam_penalty"], marker="o", label="parameter-space SAM penalty")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("representation scale s")
plt.ylabel("max nearby loss")
plt.title("Parameter-space SAM penalty depends on representation")
plt.grid(True, which="both")
plt.legend()

fig_path = FIGURES / "43_sam_penalty_vs_scale.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()
print("Saved:", fig_path)

## 4. Same radius, different function change

A fixed coordinate-space SAM radius can induce different function-space changes.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results["scale_s"], results["max_function_change_from_u_radius"], marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("representation scale s")
plt.ylabel("max effective |Δw| from fixed |εu|")
plt.title("Same parameter radius produces different function changes")
plt.grid(True, which="both")

fig_path = FIGURES / "43_same_radius_different_function_change.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()
print("Saved:", fig_path)

## 5. Parameter-space versus function-space perturbations

Compare a fixed \(u\)-space perturbation radius with a fixed effective function-space perturbation radius.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results["scale_s"], results["parameter_space_sam_penalty"], marker="o", label="fixed parameter-space radius")
plt.plot(results["scale_s"], results["function_space_penalty"], marker="o", label="fixed function-space radius")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("representation scale s")
plt.ylabel("max nearby loss")
plt.title("Parameter-space SAM versus function-space robustness")
plt.grid(True, which="both")
plt.legend()

fig_path = FIGURES / "43_parameter_vs_function_space_sam.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()
print("Saved:", fig_path)

## 6. Interpretation shift

This does not show that SAM is useless.

It shows that this inference is too fast:

\[
\text{SAM works} \Rightarrow \text{parameter-space flatness causes generalization}
\]

A more cautious interpretation is:

\[
\text{SAM works} \Rightarrow \text{some robustness proxy may help}
\]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis("off")

ax.text(0.25, 0.78, "Classical read", ha="center", fontsize=18)
ax.text(0.25, 0.52, "SAM\n↓\nflatter minima\n↓\ngeneralization", ha="center", va="center", fontsize=16)

ax.text(0.75, 0.78, "Reframed read", ha="center", fontsize=18)
ax.text(0.75, 0.52, "SAM\n↓\nrobustness proxy\n↓\ninvariant generalization?", ha="center", va="center", fontsize=16)

ax.text(0.5, 0.12, "SAM can be useful without making parameter-space flatness causal.", ha="center", fontsize=16)

fig_path = FIGURES / "43_sam_interpretation_shift.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()
print("Saved:", fig_path)

## 7. Save results

In [ ]:
csv_path = RESULTS / "43_sam_vs_representation_summary.csv"
json_path = RESULTS / "43_sam_vs_representation_summary.json"

results.to_csv(csv_path, index=False)

summary = {
    "title": "SAM vs Representation",
    "rho_u": float(rho_u),
    "delta_w": float(delta_w),
    "min_parameter_space_sam_penalty": float(results["parameter_space_sam_penalty"].min()),
    "max_parameter_space_sam_penalty": float(results["parameter_space_sam_penalty"].max()),
    "function_space_penalty": float(results["function_space_penalty"].iloc[0]),
    "max_penalty_ratio_parameter_to_function": float(results["penalty_ratio_parameter_to_function"].max()),
    "engineering_statement": "SAM can be useful without making parameter-space flatness causal. Robustness should be measured where the computation lives."
}

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)
print(summary)

## Notebook takeaway

Parameter-space SAM depends on how the model is represented.

Function-space robustness is closer to what the computation actually does.

> SAM can be useful without making parameter-space flatness causal.
>
> Robustness should be measured where the computation lives.

In [ ]:
output_files = [
    FIGURES / "43_sam_penalty_vs_scale.png",
    FIGURES / "43_same_radius_different_function_change.png",
    FIGURES / "43_parameter_vs_function_space_sam.png",
    FIGURES / "43_sam_interpretation_shift.png",
    RESULTS / "43_sam_vs_representation_summary.csv",
    RESULTS / "43_sam_vs_representation_summary.json",
]

zip_path = DOWNLOADS / "43_sam_vs_representation_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in output_files:
        if f.exists():
            z.write(f, arcname=f.relative_to(ROOT))

print("Created:", zip_path)
print("\nIncluded files:")
for f in output_files:
    print("-", f.relative_to(ROOT), "exists=" + str(f.exists()))

## Download outputs

In Colab, run the next cell to download the ZIP directly.

In local Jupyter, open:

`downloads/43_sam_vs_representation_outputs.zip`

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))
    for f in output_files:
        if f.exists():
            display(FileLink(str(f)))